In [ ]:
from platform import python_version
print(python_version())

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as npmtd
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

In [ ]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(mtd.echo_parameters())

### Get all programs

In [ ]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

#--------- chose a disease --------------------
DISEASE_ID = 'ACC'
DISEASE_ID = 'PAAD'

In [ ]:
verbose=False
force=False

method='deseq2'
imax_tumor=250
imax_normal=50

gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

exclude_prog_list=['CCLE']

dfn_tumor, dfn_normal, df_ana = gdc.get_all_data_from_disease(disease_id=DISEASE_ID, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

df_ana

In [ ]:
dfn_tumor.head(3)

In [ ]:
dfn_normal.head(3)

### Clusterization

In [ ]:
group = 'Normal'
group = 'Tumor'


disease_id=DISEASE_ID
imax_tumor=250
imax_normal=50
exclude_prog_list=['CCLE']
n_components = 10
min_clusters = 6; max_clusters = 12
n_umap_neighbors=5; min_umap_dist=0.2; umap_metric="euclidean"
method_hca="ward"; hca_criterion="maxclust"
LFC_cutoff=1; FDR_cutoff=0.05
perc_min_samples=0.25; top_n=10_000

if group == 'Tumor':
    df = dfn_tumor
    n_clusters = 10
else:
    df = dfn_normal
    n_clusters = 3


df_cluster, df_sel, df_cpm, df_pca, df_umap = gdc.cluster_expression_data_group(df, group=group, n_clusters=n_clusters, 
                                                n_components=n_components, min_clusters=min_clusters, max_clusters=max_clusters,
                                                n_umap_neighbors=n_umap_neighbors, min_umap_dist=min_umap_dist, umap_metric=umap_metric,
                                                method_hca=method_hca, hca_criterion=hca_criterion,
                                                LFC_cutoff=LFC_cutoff, FDR_cutoff=FDR_cutoff,
                                                perc_min_samples=perc_min_samples, top_n=top_n, verbose=verbose
                                                )

In [ ]:
df_cluster = gdc.write_clusters(gdc.dfall, gdc.dfsig, LFC_cutoff=LFC_cutoff, FDR_cutoff=FDR_cutoff, verbose=verbose)
df_cluster

In [ ]:
cluster_list = np.unique(gdc.dfall.cluster)
cluster_list

In [ ]:
dic = {}

for ncluster in cluster_list:
    df2 = gdc.dfsig[gdc.dfsig.cluster == ncluster]
    df2 = df2[ (df2['lfc'].abs() > LFC_cutoff) & (df2['fdr'] < FDR_cutoff) ]

    symbols = np.unique(df2.symbol)

    dic[ncluster] = set(symbols)

    print(f"Cluster {ncluster}: {len(symbols)} signature genes")

    fname = f"cluster_{ncluster}-{n_clusters}_{group}_signature_genes.txt"
    filename = gdc.root_mprog_lfc / fname

    write_txt('\n'.join(symbols), filename, verbose=verbose)
            

df_cluster['n_unique_genes']  = 0
df_cluster['unique_genes'] = None

for nclu in cluster_list:
    set0 = dic[nclu]

    for key, setx in dic.items():
        if key != nclu:
            set0 = set0 - dic[key]

    uniq_list = np.unique(list(set0))

    # LFC and DEGs only after clusterization
    df_cluster.loc[df_cluster['ncluster'] == nclu, 'n_unique_genes'] = len(uniq_list)
    df_cluster.loc[df_cluster['ncluster'] == nclu, 'unique_genes'] = "; ".join(uniq_list)

In [ ]:
uniq_list

In [ ]:
gdc.df_cpm.shape, gdc.df_sel.shape

In [ ]:
fname = f"{group}_expression_log_CPM_top_{top_n}_genes_all_samples_transposed.tsv"
pdwritecsv(df_sel, fname, gdc.root_mprog_lfc, verbose=True)

print(df_sel.shape)
df_sel.head(6)


In [ ]:
df_pca = gdc.calc_PCA(df_scaled, n_components=10, verbose=False)
df_pca.shape

In [ ]:
df_eval, df_samp_clusters = gdc.calc_best_cluster(df_pca=df_pca, min_clusters = 6, max_clusters = 12)
df_eval

In [ ]:
df_samp_clusters

In [ ]:
df_samp_clusters.groupby('cluster').size()

In [ ]:
gdc.plot_PCA(df_pca, group=group, figsize=(12, 12))

### UMAP

In [ ]:
n_umap_neighbors=5
min_umap_dist=0.2
umap_metric="euclidean"

df_umap = gdc.calc_PCA_UMAP(df_pca=df_pca, df_samp_clusters=df_samp_clusters, n_neighbors=n_umap_neighbors, min_dist=min_umap_dist, metric=umap_metric)

df_umap

In [ ]:
df_umap.groupby('cluster').size()

In [ ]:
gdc.plot_HCA_PCA(df_pca, figsize=(15,20))

### Cut tree into k clusters:

In [ ]:
method_hca = "ward"
hca_criterion="maxclust"
verbose = False

n_clusters=10


df_hca = gdc.cut_HCA_PCA(df_pca=df_pca, method=method_hca, n_clusters=n_clusters, criterion=hca_criterion, verbose=verbose)

fname = f"clusters_{n_clusters}_for_{group}_samples.tsv"
pdwritecsv(df_hca, fname, gdc.root_mprog_lfc, verbose=True)

df_hca

In [ ]:
df_hca.groupby('cluster').size()

In [ ]:
dfclu_samp = df_hca.groupby('cluster').size().reset_index(name='samples')
dfclu_samp

In [ ]:
dfc_log = np.log2(df_cpm + 1)

gene_annot = (
    dfg_filt[["geneid", "symbol"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dfall, dfsig = gdc.find_cluster_signature_genes(df_logcpm=dfc_log, df_samp_clusters=df_hca, gene_annot=gene_annot)

In [ ]:
print(len(dfall))
dfall.head(3)

In [ ]:
fname = f"{group}_clusters_{n_clusters}_signatures_up_down_DEGs.tsv"
pdwritecsv(dfall, fname, gdc.root_mprog_lfc)

In [ ]:
lfc = pd.to_numeric(dfall["lfc"], errors="coerce").dropna().to_list()

gdc.plot_histogram(data=lfc, title="Distribution of DEG log2 Fold Changes", bins=50, 
                   xlabel="log2 Fold Change", ylabel="Number of genes",
                   figsize=(9, 5))

In [ ]:
import matplotlib.pyplot as plt

lfc = pd.to_numeric(dfall["lfc"], errors="coerce").dropna()

plt.figure(figsize=(9, 5))
plt.hist(lfc, bins=60, edgecolor="black", alpha=0.8)
plt.axvline(0, linestyle="--", linewidth=1)
plt.axvline(-1, linestyle=":", linewidth=1)
plt.axvline(1, linestyle=":", linewidth=1)

plt.xlabel("log2 Fold Change")
plt.ylabel("Number of genes")
plt.title("Distribution of DEG log2 Fold Changes")
plt.tight_layout()
plt.show()

In [ ]:
print(len(dfsig))
dfsig.head(3)

In [ ]:
lfc = pd.to_numeric(dfsig["lfc"], errors="coerce").dropna().to_list()

gdc.plot_histogram(data=lfc, title="Distribution of DEG log2 Fold Changes", bins=50, 
                   xlabel="log2 Fold Change", ylabel="Number of genes",
                   figsize=(9, 5))

In [ ]:

lfc = pd.to_numeric(dfsig["lfc"], errors="coerce").dropna()

plt.figure(figsize=(9, 5))
plt.hist(lfc, bins=60, edgecolor="black", alpha=0.8)
plt.axvline(0, linestyle="--", linewidth=1)
plt.axvline(-1, linestyle=":", linewidth=1)
plt.axvline(1, linestyle=":", linewidth=1)

plt.xlabel("log2 Fold Change")
plt.ylabel("Number of genes")
plt.title("Distribution of DEG log2 Fold Changes")
plt.tight_layout()
plt.show()

In [ ]:
cluster_list = np.unique(dfall.cluster)
cluster_list

In [ ]:
LFC_cutoff=2
FDR_cutoff=1e-3

ncluster = 1

df2 = dfsig[dfsig.cluster == ncluster]
df2 = df2[ (df2['lfc'].abs() >= LFC_cutoff) & (df2['fdr'] < FDR_cutoff) ]
print(len(df2))
df2.head(3)

In [ ]:
dic = {}

for ncluster in cluster_list:
    df2 = dfsig[dfsig.cluster == ncluster]
    df2 = df2[ (df2['lfc'].abs() > LFC_cutoff) & (df2['fdr'] < FDR_cutoff) ]

    symbols = np.unique(df2.symbol)

    dic[ncluster] = set(symbols)

    print(f"Cluster {ncluster}: {len(symbols)} signature genes")

    fname = f"cluster_{ncluster}-{n_clusters}_signature_genes.txt"
    filename = gdc.root_mprog_lfc / fname

    write_txt('\n'.join(symbols), filename)


In [ ]:
df_cluster = gdc.write_clusters(dfall, dfsig, LFC_cutoff=LFC_cutoff, FDR_cutoff=FDR_cutoff, verbose=False)

df_cluster

In [ ]:
dic2 = {}

df_cluster['n_unique_genes']  = 0
df_cluster['unique_genes'] = None


for nclu in df_cluster['ncluster']:
    set0 = dic[nclu]

    for key, setx in dic.items():
        if key != nclu:
            set0 = set0 - dic[key]

    uniq_list = list(set0)
    s_lista = "; ".join(uniq_list)

    df_cluster.loc[df_cluster['ncluster'] == nclu, 'n_unique_genes'] = len(uniq_list)
    df_cluster.loc[df_cluster['ncluster'] == nclu, 'unique_genes'] = s_lista



df_cluster

In [ ]:
fname = f"clusters_{n_clusters}.tsv"
pdwritecsv(df_cluster, fname, gdc.root_mprog_lfc)



In [ ]:
gdc.root_mprog_lfc

### UMAP

In [ ]:
gdc.plot_PCA_UMAP(df_umap=df_umap, n_neighbors=n_neighbors, min_dist=min_dist, figsize=(12, 12))

In [ ]:
gdc.plot_HCA_PCA_UMAP(df_umap, figsize=(12, 8))

### Loop over clusters

In [ ]:
verbose=False
force=False

method='deseq2'
imax_tumor=250
imax_normal=50

gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

case = case_list[0]
print(f"case = {case}")

df_psi = gdc.open_primary_sites_cbio(verbose=False)


df_psi = df_psi[ (df_psi.disease_id == DISEASE_ID) & (~pd.isnull(df_psi.primary_site)) & (~pd.isnull(df_psi.cbioportal_study_id)) ].copy()
dfa = df_psi.groupby(['prog_id', 'psi_id', 'disease_id', 'primary_site', 'cbioportal_study_id']).size().reset_index()

dic = {}

for ipsi, row in dfa.iterrows():
    prog_id = row.prog_id

    if prog_id == 'CCLE':
        continue

    psi_id = row.psi_id
    disease_id = row.disease_id
    primary_site = row.primary_site

    print(f"{int(ipsi)+1}) prog_id {prog_id}, psi_id {psi_id}, primary_site {primary_site}, disease_id {disease_id}", end=" ")

    _ = gdc.get_primary_sites(prog_id=prog_id, verbose=verbose)

    ret = gdc.set_primary_site(psi_id = psi_id)

    if not ret:
        print(f"Error: failed to set primary site for {prog_id} {psi_id}")
        continue

    df_lfc = pd.DataFrame()

    # print(gdc.root_project, gdc.root_disease, '\n')

    df_tumor, df_normal, df_gtex_ctrl = gdc.calc_file_expression_tumor_normal_gtex(imax_tumor=imax_tumor, imax_normal=imax_normal, force=force, verbose=verbose)
    print(f"df_tumor {df_tumor.shape}")

    df_degs, df_lfc, degs_txt, msg = gdc.calc_degs(
        prog_id = prog_id,
        psi_id = psi_id,
        root_src = gdc.root_src,
        run_conda = True,
        lfc_cutoff = 1.0,
        fdr_cutoff = 0.05,
        method = method,
        imax_tumor = imax_tumor,
        imax_normal = imax_normal,
        force = force,
        verbose = verbose,
    )

    if not df_tumor.empty:
        dic[ipsi] ={}

        dic[ipsi]['primary_site'] = [prog_id, psi_id, disease_id, primary_site]
        dic[ipsi]['df_tumor'] = df_tumor
        dic[ipsi]['degs'] = df_degs
        dic[ipsi]['lfc'] = df_lfc

        if not df_normal.empty:
            dic[ipsi]['df_normal'] = df_normal

        if not df_gtex_ctrl.empty:
            dic[ipsi]['df_gtex'] = df_gtex_ctrl

        print(f"Tumor samples: {df_tumor.shape[1]}, Normal samples: {df_normal.shape[1]}, GTEx controls: {df_gtex_ctrl.shape[1]}\n")

    print("")

print("\n--------------- end ---------------")

